# E-08 · CatBoost residual + embedding da thumbnail

Notebook unico do experimento E-08. Ele roda inteiro no Colab, de cima para baixo (**Ambiente de execucao > Executar tudo**).

**Pergunta:** um modelo nao linear (CatBoost) prevendo o desvio em relacao a mediana do canal, com features da thumbnail e um embedding visual (SigLIP 2), supera o B0 e o R_M2 (referencia do E-07)?

**Protocolo (igual ao E-00..E-07):**
- Gold `gold_v1_20261002T013000Z_663f502d`, somente linhas de treino. **O teste final continua fechado.**
- Rolling-origin com 3 origens expansivas (50%, 65% e 80% de cada canal).
- Hiperparametros escolhidos por busca aleatoria em um holdout cronologico **dentro** de cada janela de treino; o PCA do embedding tambem e ajustado so no treino de cada janela.
- Metrica principal: MAE em `log(1 + curtidas)` medio por canal. Comparacoes pareadas por canal com bootstrap e permutacao de sinais.

**Novidades em relacao ao E-07:**
1. CatBoost com perda absoluta (MAE) no alvo residual `target_log_likes - baseline_log_likes`.
2. Embedding SigLIP 2 da thumbnail (painel vertical ja recortado), reduzido por PCA.
3. Intervalo de predicao de 80% (CatBoost MultiQuantile 10%/50%/90%) e sua cobertura.
4. Acuracia por pares **justa**: quando o modelo preve empate, conta 0,5 (o B0 preve valores iguais dentro do canal e era punido no criterio antigo). O criterio antigo tambem e reportado para comparabilidade.
5. Correlacao de Spearman dentro do canal e SHAP do modelo final.

**Entrada:** `e08_bundle.tar.gz` (Gold + manifest + 3.004 thumbnails recortadas e redimensionadas para 256 px). O notebook procura o arquivo (ou suas partes `.part00`, `.part01`...) em `/content`, depois em `MyDrive/case_galaxies/` e, se nao achar, abre o seletor de upload.

**Ambiente recomendado:** GPU T4 (*Ambiente de execucao > Alterar o tipo de ambiente*). Sem GPU tambem roda; so o embedding fica mais lento (~5-10 min).

## 1. Configuracao

In [ ]:
import os

SEED = 42
N_TRIALS = 12                 # tentativas de busca aleatoria por braco e por origem
PCA_DIM = 16                  # dimensoes do embedding apos PCA
EMB_MODEL = "google/siglip2-base-patch16-224"
EMB_FALLBACK = ("ViT-B-32", "laion400m_e32")   # open_clip, se o SigLIP 2 falhar
INTERVAL_ALPHAS = (0.1, 0.5, 0.9)              # intervalo de 80%
BOOTSTRAP_SAMPLES = 10_000
ROLLING_ORIGIN_FRACTIONS = (0.50, 0.65, 0.80)
BUNDLE_NAME = "e08_bundle.tar.gz"
WORK = os.environ.get("E08_WORK", "/content/e08")
OUT = os.path.join(WORK, "outputs")
FAKE_EMB = os.environ.get("E08_FAKE_EMB") == "1"   # so para teste local sem acesso a modelos

## 2. Dependencias

In [ ]:
import importlib, subprocess, sys
for package in ("catboost",):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import itertools, json, tarfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor, Pool
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

rng_global = np.random.default_rng(SEED)
os.makedirs(OUT, exist_ok=True)
print("catboost", __import__("catboost").__version__)

## 3. Dados

In [ ]:
def find_bundle():
    candidates = [os.path.join("/content", BUNDLE_NAME), os.path.join(WORK, BUNDLE_NAME)]
    import glob
    parts = sorted(glob.glob(os.path.join("/content", BUNDLE_NAME + ".part*")))
    if parts and not os.path.exists(candidates[0]):   # pacote enviado em partes de 9 MB
        with open(candidates[0], "wb") as joined:
            for part in parts:
                joined.write(open(part, "rb").read())
        print(f"{len(parts)} partes unidas em {candidates[0]}")
    for path in candidates:
        if os.path.exists(path):
            return path
    try:
        from google.colab import drive, files
    except ImportError:
        raise FileNotFoundError(f"Coloque {BUNDLE_NAME} em {candidates[0]}")
    drive_path = f"/content/drive/MyDrive/case_galaxies/{BUNDLE_NAME}"
    if not os.path.exists("/content/drive"):
        try:
            drive.mount("/content/drive")
        except Exception as error:
            print("Drive nao montado:", error)
    if os.path.exists(drive_path):
        return drive_path
    print(f"Selecione {BUNDLE_NAME} no seu computador")
    uploaded = files.upload()
    name = next(iter(uploaded))
    os.replace(name, candidates[0])
    return candidates[0]

bundle_dir = os.path.join(WORK, "bundle")
if not os.path.exists(os.path.join(bundle_dir, "training_snapshot.csv")):
    bundle = find_bundle()
    with tarfile.open(bundle) as tar:
        tar.extractall(WORK)
manifest = json.load(open(os.path.join(bundle_dir, "manifest.json"), encoding="utf-8"))
gold = pd.read_csv(os.path.join(bundle_dir, "training_snapshot.csv"), parse_dates=["published_at"])
print(manifest["dataset_version"], gold.shape)

TARGET = "target_log_likes"
BASELINE = "baseline_log_likes"
data = gold.loc[gold["split"].eq("train")].reset_index(drop=True)   # teste final fechado
data["thumb_ocr_log_word_count"] = np.log1p(data["thumb_ocr_word_count"].fillna(0))
coverage = {
    "rows": int(len(data)), "channels": int(data["channel_id"].nunique()),
    "sectors": data["sector"].value_counts().to_dict(),
    "channel_types": data["channel_type"].value_counts().to_dict(),
}
print(json.dumps(coverage, indent=2, ensure_ascii=False))

### Grupos de features

Mesmas definicoes dos experimentos anteriores. `BASE` e o conjunto do E-07; `THUMB` sao as features manuais de thumbnail e OCR (E-06); `EMB` e o embedding novo; `TEXT` e o subgrupo do E-02.

In [ ]:
BASE_NUM = ["duration_seconds", "channel_prior_posts", "baseline_log_likes", "days_since_previous_post",
            "title_length", "description_length", "tag_count"]
BASE_CAT = ["channel_id", "sector", "channel_type", "publish_hour_utc", "publish_weekday"]
THUMB_NUM = ["thumb_is_vertical_pillarbox", "thumb_letterbox_fraction", "thumb_is_low_resolution",
             "thumb_brightness_mean", "thumb_brightness_std", "thumb_saturation_mean", "thumb_colorfulness",
             "thumb_dark_pixel_pct", "thumb_light_pixel_pct", "thumb_edge_density", "thumb_has_text",
             "thumb_ocr_log_word_count", "thumb_ocr_text_area_pct", "thumb_ocr_uppercase_ratio",
             "thumb_ocr_has_digits", "thumb_ocr_has_question", "thumb_ocr_has_exclamation"]
THUMB_CAT = ["thumb_dominant_hue"]
TEXT_NUM = ["has_cta", "title_tag_overlap"]
TEXT_CAT = ["duration_bucket"]
EMB_COLS = [f"emb_pc{i + 1:02d}" for i in range(PCA_DIM)]

ARMS = {   # nome: (numericas, categoricas, usa_embedding)
    "CB_BASE":  (BASE_NUM, BASE_CAT, False),
    "CB_THUMB": (BASE_NUM + THUMB_NUM, BASE_CAT + THUMB_CAT, False),
    "CB_EMB":   (BASE_NUM, BASE_CAT, True),
    "CB_FULL":  (BASE_NUM + THUMB_NUM + TEXT_NUM, BASE_CAT + THUMB_CAT + TEXT_CAT, True),
}
INTERVAL_ARMS = ("CB_BASE", "CB_FULL")
missing = [c for c in set(sum([a[0] + a[1] for a in ARMS.values()], [])) if c not in data.columns]
assert not missing, missing

## 4. Embedding SigLIP 2 da thumbnail

In [ ]:
emb_cache = os.path.join(WORK, "thumbnail_embeddings.npz")

def load_images(ids):
    from PIL import Image
    return [Image.open(os.path.join(bundle_dir, "thumbs", f"{i}.jpg")).convert("RGB") for i in ids]

def embed_siglip(ids, batch=64):
    import torch
    from transformers import AutoModel, AutoProcessor
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()
    processor = AutoProcessor.from_pretrained(EMB_MODEL)
    chunks = []
    for start in range(0, len(ids), batch):
        inputs = processor(images=load_images(ids[start:start + batch]), return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.get_image_features(**inputs)
        if not isinstance(out, torch.Tensor):
            out = getattr(out, "pooler_output", None) if getattr(out, "pooler_output", None) is not None else out[0]
        chunks.append(torch.nn.functional.normalize(out.float(), dim=-1).cpu().numpy())
        print(f"\r{min(start + batch, len(ids))}/{len(ids)}", end="")
    print()
    return np.vstack(chunks), EMB_MODEL

def embed_open_clip(ids, batch=64):
    import torch, open_clip
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, _, preprocess = open_clip.create_model_and_transforms(EMB_FALLBACK[0], pretrained=EMB_FALLBACK[1])
    model = model.to(device).eval()
    chunks = []
    for start in range(0, len(ids), batch):
        images = torch.stack([preprocess(im) for im in load_images(ids[start:start + batch])]).to(device)
        with torch.no_grad():
            out = model.encode_image(images)
        chunks.append(torch.nn.functional.normalize(out.float(), dim=-1).cpu().numpy())
    return np.vstack(chunks), "open_clip:" + ":".join(EMB_FALLBACK)

ids = gold["video_id"].tolist()
if os.path.exists(emb_cache):
    cached = np.load(emb_cache, allow_pickle=True)
    embeddings, emb_source = cached["embeddings"], str(cached["source"])
elif FAKE_EMB:
    embeddings, emb_source = rng_global.normal(size=(len(ids), 64)), "fake"
else:
    t0 = time.time()
    try:
        embeddings, emb_source = embed_siglip(ids)
    except Exception as error:
        print("SigLIP 2 falhou, usando open_clip:", repr(error))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
        embeddings, emb_source = embed_open_clip(ids)
    print(f"{emb_source}: {embeddings.shape} em {time.time() - t0:.0f}s")
    np.savez(emb_cache, embeddings=embeddings, video_id=np.array(ids), source=emb_source)

emb_by_id = dict(zip(ids, embeddings))
E = np.vstack([emb_by_id[v] for v in data["video_id"]])
print(emb_source, E.shape)

## 5. Protocolo (copiado de `scripts/run_baseline_experiments.py` e `run_residual_experiment.py`)

In [ ]:
def rolling_origin_splits(frame):
    folds = []
    for position, fraction in enumerate(ROLLING_ORIGIN_FRACTIONS):
        nxt = ROLLING_ORIGIN_FRACTIONS[position + 1] if position + 1 < len(ROLLING_ORIGIN_FRACTIONS) else 1.0
        fit_idx, val_idx = [], []
        for _, channel in frame.sort_values(["channel_id", "published_at", "video_id"]).groupby("channel_id"):
            train_end = max(5, int(np.floor(len(channel) * fraction)))
            val_end = min(max(train_end + 1, int(np.floor(len(channel) * nxt))), len(channel))
            fit_idx.extend(channel.index[:train_end])
            val_idx.extend(channel.index[train_end:val_end])
        folds.append((f"origin_{position + 1}", frame.loc[fit_idx].copy(), frame.loc[val_idx].copy()))
    return folds

def validation_split(frame):
    dev_idx, hold_idx = [], []
    for _, channel in frame.sort_values(["channel_id", "published_at"]).groupby("channel_id"):
        size = max(1, int(np.ceil(len(channel) * 0.20)))
        dev_idx.extend(channel.index[:-size])
        hold_idx.extend(channel.index[-size:])
    return frame.loc[dev_idx].copy(), frame.loc[hold_idx].copy()

def mae_by_channel(frame, predictions):
    err = np.abs(frame[TARGET].to_numpy() - predictions)
    return float(pd.Series(err, index=frame.index).groupby(frame["channel_id"]).mean().mean())

# ---- R_M2: Ridge-LAD residual por tipo de canal (referencia do E-07) ----
ALPHAS = (0.1, 1.0, 10.0, 100.0)

class RidgeTabular:
    def __init__(self, alpha, cat, num):
        self.alpha, self.cat, self.num = alpha, cat, num
    def _prepare(self, frame):
        self.means = frame[self.num].mean()
        self.scales = frame[self.num].std().replace(0, 1).fillna(1)
        self.cats = {c: sorted(frame[c].fillna("<missing>").astype(str).unique()) for c in self.cat}
    def matrix(self, frame):
        parts = [np.ones((len(frame), 1)),
                 frame[self.num].fillna(self.means).sub(self.means).div(self.scales).to_numpy(float)]
        for c in self.cat:
            values = frame[c].fillna("<missing>").astype(str).to_numpy()
            parts.append(np.column_stack([(values == v).astype(float) for v in self.cats[c]]))
        return np.hstack(parts)
    def fit_lad(self, frame, target, iterations=30):
        self._prepare(frame)
        X, y = self.matrix(frame), target
        penalty = np.eye(X.shape[1]); penalty[0, 0] = 0
        w = np.ones(len(y))
        for _ in range(iterations):
            Xw = X * w[:, None]
            self.coef = np.linalg.solve(X.T @ Xw + self.alpha * penalty, Xw.T @ y)
            w = 1.0 / np.maximum(np.abs(y - X @ self.coef), 1e-3)
        return self
    def predict(self, frame):
        return self.matrix(frame) @ self.coef

def lad_fit_predict(fit, val, num, cat):
    dev, hold = validation_split(fit)
    resid = lambda f: (f[TARGET] - f[BASELINE]).to_numpy()
    scores = {a: mae_by_channel(hold, hold[BASELINE].to_numpy() + RidgeTabular(a, cat, num).fit_lad(dev, resid(dev)).predict(hold))
              for a in ALPHAS}
    alpha = min(ALPHAS, key=scores.get)
    return val[BASELINE].to_numpy() + RidgeTabular(alpha, cat, num).fit_lad(fit, resid(fit)).predict(val)

def r_m2(fit, val):
    cat = [c for c in BASE_CAT if c != "channel_type"]
    out = np.empty(len(val))
    for kind in sorted(fit["channel_type"].unique()):
        mask = val["channel_type"].eq(kind).to_numpy()
        if mask.any():
            out[mask] = lad_fit_predict(fit.loc[fit["channel_type"].eq(kind)], val.loc[mask], BASE_NUM, cat)
    return out

### CatBoost residual

O alvo e o desvio em relacao ao B0 e a perda e absoluta (alinhada ao MAE). Para cada braco e cada origem: busca aleatoria de `N_TRIALS` combinacoes no holdout cronologico interno (com *early stopping*), escolha pelo MAE medio por canal, e reajuste em toda a janela de treino com o numero de iteracoes encontrado.

In [ ]:
SEARCH_SPACE = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "l2_leaf_reg": [1, 3, 5, 10, 20],
    "random_strength": [0.1, 0.5, 1, 2],
    "bagging_temperature": [0, 0.5, 1, 2],
}

def sample_params(rng, n):
    keys = list(SEARCH_SPACE)
    seen, out = set(), []
    while len(out) < n:
        p = {k: SEARCH_SPACE[k][rng.integers(len(SEARCH_SPACE[k]))] for k in keys}
        key = tuple(p.values())
        if key not in seen:
            seen.add(key); out.append(p)
    return out

def add_embedding(fit, val, use_emb):
    if not use_emb:
        return fit, val
    pca = PCA(n_components=PCA_DIM, random_state=SEED).fit(E[fit.index])
    fit, val = fit.copy(), val.copy()
    fit[EMB_COLS] = pca.transform(E[fit.index])
    val[EMB_COLS] = pca.transform(E[val.index])
    return fit, val

def to_pool(frame, num, cat, label=True):
    X = frame[num + cat].copy()
    for c in cat:
        X[c] = X[c].fillna("<missing>").astype(str)
    y = (frame[TARGET] - frame[BASELINE]).to_numpy() if label else None
    return Pool(X, y, cat_features=cat)

def catboost_arm(fit, val, num, cat, use_emb, rng, with_interval):
    fit, val = add_embedding(fit, val, use_emb)
    num = num + (EMB_COLS if use_emb else [])
    dev, hold = validation_split(fit)
    dev_pool, hold_pool = to_pool(dev, num, cat), to_pool(hold, num, cat)
    trials = []
    for params in sample_params(rng, N_TRIALS):
        model = CatBoostRegressor(loss_function="MAE", iterations=3000, od_type="Iter", od_wait=150,
                                  random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=-1, **params)
        model.fit(dev_pool, eval_set=hold_pool, use_best_model=True)
        score = mae_by_channel(hold, hold[BASELINE].to_numpy() + model.predict(hold_pool))
        trials.append((score, params, max(50, model.get_best_iteration() + 1)))
    score, params, best_iter = min(trials, key=lambda t: t[0])
    iterations = int(best_iter * 1.1)
    fit_pool, val_pool = to_pool(fit, num, cat), to_pool(val, num, cat, label=False)
    final = CatBoostRegressor(loss_function="MAE", iterations=iterations, random_seed=SEED, verbose=False,
                              allow_writing_files=False, thread_count=-1, **params).fit(fit_pool)
    result = {"pred": val[BASELINE].to_numpy() + final.predict(val_pool),
              "params": {**params, "iterations": iterations, "holdout_macro_mae": score}}
    if with_interval:
        alphas = ",".join(str(a) for a in INTERVAL_ALPHAS)
        q = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", iterations=iterations, random_seed=SEED,
                              verbose=False, allow_writing_files=False, thread_count=-1, **params).fit(fit_pool)
        quantiles = np.sort(q.predict(val_pool), axis=1) + val[BASELINE].to_numpy()[:, None]
        result["low"], result["high"] = quantiles[:, 0], quantiles[:, -1]
    return result

## 6. Rolling-origin

In [ ]:
rng = np.random.default_rng(SEED)
predictions, chosen_params = [], []
t0 = time.time()
for origin, fit, val in rolling_origin_splits(data):
    frame = val[["video_id", "channel_id", "channel_name", "channel_type", "sector", "published_at", TARGET, BASELINE]].copy()
    frame["origin"] = origin
    frame["pred_B0"] = val[BASELINE].to_numpy()
    frame["pred_R_M2"] = r_m2(fit, val)
    for arm, (num, cat, use_emb) in ARMS.items():
        result = catboost_arm(fit, val, num, cat, use_emb, rng, arm in INTERVAL_ARMS)
        frame[f"pred_{arm}"] = result["pred"]
        if "low" in result:
            frame[f"low_{arm}"], frame[f"high_{arm}"] = result["low"], result["high"]
        chosen_params.append({"origin": origin, "arm": arm, **result["params"]})
        print(f"{origin} {arm} ok ({time.time() - t0:.0f}s)", flush=True)
    predictions.append(frame)
predictions = pd.concat(predictions, ignore_index=True)
chosen_params = pd.DataFrame(chosen_params)
MODELS = ["B0", "R_M2"] + list(ARMS)

## 7. Metricas

- `mae`: MAE em log por canal (metrica principal).
- `pair_old`: acuracia por pares com o criterio antigo (empate previsto conta como erro), para comparar com E-00..E-07.
- `pair_fair`: empate previsto conta 0,5. **Esta e a leitura correta para comparar com o B0.**
- `spearman`: correlacao de postos dentro do canal (0 quando a previsao e constante).

In [ ]:
def pairwise(actual, predicted, tie_value):
    left, right = np.triu_indices(len(actual), k=1)
    a = np.sign(actual[left] - actual[right])
    p = np.sign(predicted[left] - predicted[right])
    ok = a != 0
    if not ok.any():
        return np.nan
    score = np.where(p[ok] == 0, tie_value, (a[ok] == p[ok]).astype(float))
    return float(score.mean())

rows = []
for (origin, channel), g in predictions.groupby(["origin", "channel_name"]):
    actual = g[TARGET].to_numpy()
    for m in MODELS:
        p = g[f"pred_{m}"].to_numpy()
        rho = spearmanr(actual, p).correlation if np.ptp(p) > 0 and np.ptp(actual) > 0 else 0.0
        row = {"origin": origin, "channel_name": channel, "channel_type": g["channel_type"].iat[0],
               "sector": g["sector"].iat[0], "model": m, "rows": len(g),
               "mae": np.mean(np.abs(actual - p)), "bias": np.mean(p - actual),
               "pair_old": pairwise(actual, p, 0.0), "pair_fair": pairwise(actual, p, 0.5),
               "spearman": rho}
        if f"low_{m}" in g:
            low, high = g[f"low_{m}"].to_numpy(), g[f"high_{m}"].to_numpy()
            row["coverage_80"] = np.mean((actual >= low) & (actual <= high))
            row["interval_width"] = np.mean(high - low)
        rows.append(row)
per_channel_origin = pd.DataFrame(rows)
METRICS = ["mae", "pair_old", "pair_fair", "spearman", "bias", "coverage_80", "interval_width"]
by_origin = per_channel_origin.groupby(["origin", "model"])[METRICS].mean().reset_index()
summary = by_origin.groupby("model")[METRICS].mean().reindex(MODELS).sort_values("mae")
per_channel = per_channel_origin.groupby(["model", "channel_name", "channel_type", "sector"])[METRICS].mean().reset_index()
wins = by_origin.pivot(index="origin", columns="model", values="mae")
summary["origens_vencendo_B0"] = [int((wins[m] < wins["B0"]).sum()) if m != "B0" else np.nan for m in summary.index]
display(summary.round(4))

In [ ]:
by_segment = per_channel.groupby(["model", "channel_type"])[["mae", "pair_fair", "spearman"]].mean().unstack().round(3)
display(by_segment.reindex(MODELS))
display(per_channel.groupby(["model", "sector"])[["mae", "pair_fair"]].mean().unstack().round(3).reindex(MODELS))

## 8. Comparacoes pareadas por canal

In [ ]:
def paired(candidate, reference, metric="mae"):
    values = per_channel.pivot(index="channel_name", columns="model", values=metric)
    d = (values[candidate] - values[reference]).dropna().to_numpy()
    rng_p = np.random.default_rng(SEED)
    boot = rng_p.choice(d, size=(BOOTSTRAP_SAMPLES, len(d)), replace=True).mean(axis=1)
    if len(d) <= 16:
        signs, method = np.array(list(itertools.product((-1, 1), repeat=len(d)))), "exact"
    else:
        signs, method = rng_p.choice((-1, 1), size=(BOOTSTRAP_SAMPLES, len(d))), f"monte_carlo_{BOOTSTRAP_SAMPLES}"
    perm = (signs * d).mean(axis=1)
    better = (d < 0) if metric == "mae" else (d > 0)
    return {"comparacao": f"{candidate} - {reference}", "metrica": metric, "delta": float(d.mean()),
            "ic95_low": float(np.quantile(boot, 0.025)), "ic95_high": float(np.quantile(boot, 0.975)),
            "p_valor": float((np.abs(perm) >= abs(d.mean())).mean()), "metodo": method,
            "canais_melhor": int(better.sum()), "canais_pior": int(((d > 0) if metric == "mae" else (d < 0)).sum())}

pairs = [(arm, "B0") for arm in ARMS] + [(arm, "R_M2") for arm in ARMS] + [("R_M2", "B0")]
pairs += [("CB_THUMB", "CB_BASE"), ("CB_EMB", "CB_BASE"), ("CB_FULL", "CB_BASE")]
stats = pd.DataFrame([paired(c, r, m) for m in ("mae", "pair_fair") for c, r in pairs])
display(stats.round(4))

## 9. Intervalo de predicao (80%)

In [ ]:
display(summary.loc[list(INTERVAL_ARMS), ["coverage_80", "interval_width"]].round(3))
print("Cobertura ideal: 0,80. Largura em log: um intervalo de largura w cobre um fator de exp(w) em curtidas.")
for arm in INTERVAL_ARMS:
    w = summary.loc[arm, "interval_width"]
    print(f"{arm}: largura media {w:.2f} em log  ->  fator {np.exp(w):.1f}x entre limite inferior e superior")

## 10. Modelo final e SHAP

Escolhe o braco com menor MAE medio no rolling-origin, usa os hiperparametros escolhidos na ultima origem e treina em **todas** as linhas de treino. Os valores SHAP explicam o desvio previsto em relacao a mediana do canal. As 16 componentes do embedding sao somadas em um unico grupo.

In [ ]:
best_arm = summary.drop(index=["B0", "R_M2"])["mae"].idxmin()
num, cat, use_emb = ARMS[best_arm]
params = chosen_params.query("arm == @best_arm and origin == 'origin_3'").iloc[0].to_dict()
iterations = int(params.pop("iterations")); params.pop("holdout_macro_mae"); params.pop("arm"); params.pop("origin")
params = {k: (int(v) if k == "depth" else v) for k, v in params.items()}
full = data.copy()
if use_emb:
    pca_final = PCA(n_components=PCA_DIM, random_state=SEED).fit(E)
    full[EMB_COLS] = pca_final.transform(E)
cols = num + (EMB_COLS if use_emb else [])
pool = to_pool(full, cols, cat)
final_model = CatBoostRegressor(loss_function="MAE", iterations=iterations, random_seed=SEED, verbose=False,
                                allow_writing_files=False, **params).fit(pool)
shap_values = final_model.get_feature_importance(pool, type="ShapValues")[:, :-1]
names = cols + cat
importance = pd.Series(np.abs(shap_values).mean(axis=0), index=names)
grouped = importance.groupby(lambda n: "thumb_embedding (PCA)" if n.startswith("emb_pc") else n).sum().sort_values(ascending=False)
print("Braco final:", best_arm, params, "iterations =", iterations)
ax = grouped.head(20)[::-1].plot.barh(figsize=(7, 6), title=f"SHAP medio |valor| - {best_arm}")
ax.set_xlabel("impacto medio no desvio previsto (log curtidas)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "shap_top20.png"), dpi=150); plt.show()

## 11. Graficos e exportacao

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pivot = by_origin.pivot(index="origin", columns="model", values="mae")[MODELS]
pivot.plot(marker="o", ax=axes[0], title="MAE por origem (menor e melhor)")
by_origin.pivot(index="origin", columns="model", values="pair_fair")[MODELS].plot(
    marker="o", ax=axes[1], title="Acuracia por pares justa (0,5 = moeda)")
axes[1].axhline(0.5, color="grey", lw=0.8, ls="--")
for ax in axes:
    ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "rolling_origin_e08.png"), dpi=150); plt.show()

predictions.to_csv(os.path.join(OUT, "rolling_origin_predictions.csv"), index=False)
per_channel_origin.to_csv(os.path.join(OUT, "rolling_origin_per_channel_metrics.csv"), index=False)
by_origin.to_csv(os.path.join(OUT, "rolling_origin_by_cut.csv"), index=False)
summary.to_csv(os.path.join(OUT, "rolling_origin_summary.csv"))
stats.to_csv(os.path.join(OUT, "comparison_statistics.csv"), index=False)
chosen_params.to_csv(os.path.join(OUT, "chosen_params.csv"), index=False)
grouped.rename("mean_abs_shap").to_csv(os.path.join(OUT, "shap_importance.csv"))
final_model.save_model(os.path.join(OUT, f"catboost_{best_arm}.cbm"))
run_info = {"experiment": "E-08", "dataset_version": manifest["dataset_version"], "embedding_source": emb_source,
            "pca_dim": PCA_DIM, "n_trials": N_TRIALS, "seed": SEED, "coverage": coverage, "best_arm": best_arm,
            "final_params": {**params, "iterations": iterations}, "test_split": "fechado"}
json.dump(run_info, open(os.path.join(OUT, "run_info.json"), "w"), indent=2, ensure_ascii=False, default=str)

import shutil
archive = shutil.make_archive(os.path.join(WORK, "e08_outputs"), "zip", OUT)
print("Resultados em", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## 12. Como registrar

1. Descompacte `e08_outputs.zip` em `artifacts/feature_experiments/e08_catboost_residual/`.
2. Registre o resultado em `docs/07-registro-de-features.md` (diario de experimentos) e, se mudar a referencia, em `docs/02-registro-de-descobertas.md`.
3. O teste final continua fechado: so abrir depois de congelar braco, features e hiperparametros.